# 🧹 Data Assessing and Cleaning

A practical notebook for **assessing, identifying, and cleaning data quality issues** using Pandas.

This notebook works with clinical-trial related datasets and demonstrates how to move from raw data to cleaner, more analysis-ready tables.

### Data Quality Dimensions Covered

- **Completeness** — whether required data is present
- **Validity** — whether values follow the expected format, type, or range
- **Accuracy** — whether values correctly represent the real-world information
- **Consistency** — whether the same information is represented in a uniform way
- **Uniqueness** — whether duplicate records are present
- **Tidiness** — whether each variable and observation is stored in an appropriate structure

> **Source note:** This notebook is my personal implementation and documentation of data-cleaning techniques learned through the CampusX DSMP 2 course. The explanations in this notebook are written to describe my own analysis and cleaning process.


In [1]:
import numpy as np
import pandas as pd

## 📑 Table of Contents

1. [Introduction](#-data-assessing-and-cleaning)
2. [Loading the Data](#-loading-the-data)
3. [Initial Data Assessment](#-initial-data-assessment)
4. [Identifying Data Quality Issues](#-identifying-data-quality-issues)
5. [Cleaning the Patient Data](#-cleaning-the-patient-data)
6. [Tidying and Reshaping Treatment Data](#-tidying-and-reshaping-treatment-data)
7. [Cleaning Treatment Data](#-cleaning-treatment-data)
8. [Summary](#-summary)


In [2]:
patients=pd.read_csv('/content/patients.csv')
treatments=pd.read_csv('/content/treatments.csv')
adverse_rxn=pd.read_csv('/content/adverse_reactions.csv')
treatments_cut=pd.read_csv('/content/treatments_cut.csv')

## 📂 Loading the Data

The notebook uses four CSV datasets:

- `patients.csv` — patient demographic and contact information
- `treatments.csv` — treatment and dosage information
- `adverse_reactions.csv` — recorded adverse reactions
- `treatments_cut.csv` — an additional treatment table used during the cleaning process

The first step is to load the datasets into Pandas DataFrames so they can be assessed and cleaned.


In [3]:
#for manual assesment export data to google sheets
with pd.ExcelWriter('clinical_trails.xlsx') as writer:
  patients.to_excel(writer,sheet_name='patients')
  treatments.to_excel(writer,sheet_name='treatments')
  treatments_cut.to_excel(writer,sheet_name='treatments_cut')
  adverse_rxn.to_excel(writer,sheet_name='adverse_rxn')



## 🔎 Initial Data Assessment

Data assessment is the process of inspecting a dataset before modifying it.

Here we use a combination of:

- `head()` and `tail()` to inspect sample records
- `sample()` to view a random record
- `info()` to inspect columns, missing values, and data types
- duplicate checks to identify repeated records
- `describe()` to inspect numerical distributions and suspicious values

The goal is to **identify problems first and clean them afterward**.


In [4]:
#programatic assessment
#head and tail
patients.head()

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,contact,birthdate,weight,height,bmi
0,1,female,Zoe,Wellish,576 Brown Bear Drive,Rancho California,California,92390.0,United States,951-719-9170ZoeWellish@superrito.com,7/10/1976,121.7,66,19.6
1,2,female,Pamela,Hill,2370 University Hill Road,Armstrong,Illinois,61812.0,United States,PamelaSHill@cuvox.de+1 (217) 569-3204,4/3/1967,118.8,66,19.2
2,3,male,Jae,Debord,1493 Poling Farm Road,York,Nebraska,68467.0,United States,402-363-6804JaeMDebord@gustr.com,2/19/1980,177.8,71,24.8
3,4,male,Liêm,Phan,2335 Webster Street,Woodbridge,NJ,7095.0,United States,PhanBaLiem@jourrapide.com+1 (732) 636-8246,7/26/1951,220.9,70,31.7
4,5,male,Tim,Neudorf,1428 Turkey Pen Lane,Dothan,AL,36303.0,United States,334-515-7487TimNeudorf@cuvox.de,2/18/1928,192.3,27,26.1


In [5]:
patients.tail()

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,contact,birthdate,weight,height,bmi
498,499,male,Mustafa,Lindström,2530 Victoria Court,Milton Mills,ME,3852.0,United States,207-477-0579MustafaLindstrom@jourrapide.com,4/10/1959,181.1,72,24.6
499,500,male,Ruman,Bisliev,494 Clarksburg Park Road,Sedona,AZ,86341.0,United States,928-284-4492RumanBisliev@gustr.com,3/26/1948,239.6,70,34.4
500,501,female,Jinke,de Keizer,649 Nutter Street,Overland Park,MO,64110.0,United States,816-223-6007JinkedeKeizer@teleworm.us,1/13/1971,171.2,67,26.8
501,502,female,Chidalu,Onyekaozulu,3652 Boone Crockett Lane,Seattle,WA,98109.0,United States,ChidaluOnyekaozulu@jourrapide.com1 360 443 2060,2/13/1952,176.9,67,27.7
502,503,male,Pat,Gersten,2778 North Avenue,Burr,Nebraska,68324.0,United States,PatrickGersten@rhyta.com402-848-4923,5/3/1954,138.2,71,19.3


In [6]:
patients.sample()

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,contact,birthdate,weight,height,bmi
24,25,male,Jakob,Jakobsen,648 Old Dear Lane,Port Jervis,New York,12771.0,United States,JakobCJakobsen@einrot.com+1 (845) 858-7707,8/1/1985,155.8,67,24.4


In [7]:
patients.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   patient_id    503 non-null    int64  
 1   assigned_sex  503 non-null    object 
 2   given_name    503 non-null    object 
 3   surname       503 non-null    object 
 4   address       491 non-null    object 
 5   city          491 non-null    object 
 6   state         491 non-null    object 
 7   zip_code      491 non-null    float64
 8   country       491 non-null    object 
 9   contact       491 non-null    object 
 10  birthdate     503 non-null    object 
 11  weight        503 non-null    float64
 12  height        503 non-null    int64  
 13  bmi           503 non-null    float64
dtypes: float64(3), int64(2), object(9)
memory usage: 55.1+ KB


### 🔍 First Assessment: Missing Patient Information

The output shows patients for whom the address is missing. The missing values extend across related location and contact fields.

**Data Quality Dimension:** Completeness

**Issue Type:** Dirty data

**Why it matters:** Missing values can prevent analysis that depends on location or contact information.

The cleaning strategy is considered later in the notebook rather than changing the raw data immediately.


In [8]:
patients[patients['address'].isnull()]

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,contact,birthdate,weight,height,bmi
209,210,female,Lalita,Eldarkhanov,NaN,NaN,NaN,NaN,NaN,NaN,8/14/1950,143.4,62,26.2
219,220,male,Mỹ,Quynh,NaN,NaN,NaN,NaN,NaN,NaN,4/9/1978,237.8,69,35.1
230,231,female,Elisabeth,Knudsen,NaN,NaN,NaN,NaN,NaN,NaN,9/23/1976,165.9,63,29.4
234,235,female,Martina,Tománková,NaN,NaN,NaN,NaN,NaN,NaN,4/7/1936,199.5,65,33.2
242,243,male,John,O'Brian,NaN,NaN,NaN,NaN,NaN,NaN,2/25/1957,205.3,74,26.4
249,250,male,Benjamin,Mehler,NaN,NaN,NaN,NaN,NaN,NaN,10/30/1951,146.5,69,21.6
257,258,male,Jin,Kung,NaN,NaN,NaN,NaN,NaN,NaN,5/17/1995,231.7,69,34.2
264,265,female,Wafiyyah,Asfour,NaN,NaN,NaN,NaN,NaN,NaN,11/3/1989,158.6,63,28.1
269,270,female,Flavia,Fiorentino,NaN,NaN,NaN,NaN,NaN,NaN,10/9/1937,175.2,61,33.1
278,279,female,Generosa,Cabán,NaN,NaN,NaN,NaN,NaN,NaN,12/16/1962,124.3,69,18.4


In [9]:
treatments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 280 entries, 0 to 279
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   given_name    280 non-null    object 
 1   surname       280 non-null    object 
 2   auralin       280 non-null    object 
 3   novodra       280 non-null    object 
 4   hba1c_start   280 non-null    float64
 5   hba1c_end     280 non-null    float64
 6   hba1c_change  171 non-null    float64
dtypes: float64(3), object(4)
memory usage: 15.4+ KB


In [10]:
adverse_rxn.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   given_name        34 non-null     object
 1   surname           34 non-null     object
 2   adverse_reaction  34 non-null     object
dtypes: object(3)
memory usage: 948.0+ bytes


### 🔍 Checking for Duplicate Patient Records

A duplicate check helps determine whether complete patient records appear more than once.

**Data Quality Dimension:** Uniqueness

**Issue Type:** Dirty data

A duplicate record can cause double-counting and distort analysis.


In [11]:
patients.duplicated().sum()

np.int64(0)

### 🔍 Checking Patient ID Uniqueness

`patient_id` is expected to identify a patient uniquely. Checking duplicate IDs helps verify whether the identifier is unique.

**Data Quality Dimension:** Uniqueness / Validity

A repeated identifier can make it difficult to reliably identify individual patients.


In [12]:
patients['patient_id'].duplicated().sum()

np.int64(0)

### 🔍 Duplicate Names

Here the notebook checks repeated combinations of `given_name` and `surname`.

A repeated name does **not automatically mean that two records are duplicates**, because different people can share the same name. Therefore, this check is useful for investigation, but the unique patient identifier should be considered before deleting records.

**Data Quality Dimension:** Uniqueness / Accuracy

**Issue Type:** Potential duplicate data


In [13]:
patients[patients.duplicated(subset=['given_name','surname'])]

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,contact,birthdate,weight,height,bmi
229,230,male,John,Doe,123 Main Street,New York,NY,12345.0,United States,johndoe@email.com1234567890,1/1/1975,180.0,72,24.4
237,238,male,John,Doe,123 Main Street,New York,NY,12345.0,United States,johndoe@email.com1234567890,1/1/1975,180.0,72,24.4
244,245,male,John,Doe,123 Main Street,New York,NY,12345.0,United States,johndoe@email.com1234567890,1/1/1975,180.0,72,24.4
251,252,male,John,Doe,123 Main Street,New York,NY,12345.0,United States,johndoe@email.com1234567890,1/1/1975,180.0,72,24.4
277,278,male,John,Doe,123 Main Street,New York,NY,12345.0,United States,johndoe@email.com1234567890,1/1/1975,180.0,72,24.4


In [14]:
treatments[treatments.duplicated()]

,given_name,surname,auralin,novodra,hba1c_start,hba1c_end,hba1c_change
136,joseph,day,29u - 36u,-,7.7,7.19,NaN


In [15]:
treatments[treatments.duplicated(subset=['given_name','surname'])]

,given_name,surname,auralin,novodra,hba1c_start,hba1c_end,hba1c_change
136,joseph,day,29u - 36u,-,7.7,7.19,NaN


In [16]:
treatments_cut[treatments_cut.duplicated(subset=['given_name','surname'])]

,given_name,surname,auralin,novodra,hba1c_start,hba1c_end,hba1c_change


In [17]:
adverse_rxn.duplicated().sum()

np.int64(0)

### 📊 Checking Numerical Distributions

`describe()` provides summary statistics such as count, mean, minimum, maximum, and quartiles.

This is useful for detecting **suspicious or implausible values** that may indicate data-entry errors.

**Data Quality Dimension:** Accuracy

**Issue Type:** Dirty data


In [18]:
patients.describe()

,patient_id,zip_code,weight,height,bmi
count,503.000000,491.000000,503.000000,503.000000,503.000000
mean,252.000000,49084.118126,173.434990,66.634195,27.483897
std,145.347859,30265.807442,33.916741,4.411297,5.276438
min,1.000000,1002.000000,48.800000,27.000000,17.100000
25%,126.500000,21920.500000,149.300000,63.000000,23.300000
50%,252.000000,48057.000000,175.300000,67.000000,27.200000
75%,377.500000,75679.000000,199.500000,70.000000,31.750000
max,503.000000,99701.000000,255.900000,79.000000,37.700000


In [19]:
patients[patients.weight==48.8]

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,contact,birthdate,weight,height,bmi
210,211,female,Camilla,Zaitseva,4689 Briarhill Lane,Wooster,OH,44691.0,United States,330-202-2145CamillaZaitseva@superrito.com,11/26/1938,48.8,63,19.1


In [20]:
treatments.describe()

,hba1c_start,hba1c_end,hba1c_change
count,280.000000,280.000000,171.000000
mean,7.985929,7.589286,0.546023
std,0.568638,0.569672,0.279555
min,7.500000,7.010000,0.200000
25%,7.660000,7.270000,0.340000
50%,7.800000,7.420000,0.380000
75%,7.970000,7.570000,0.920000
max,9.950000,9.580000,0.990000


## 🧹 Cleaning the Patient Data

Before making changes, copies of the original DataFrames are created.

This is a good practice because it preserves the original data and allows the cleaning process to be performed on separate working DataFrames.


In [21]:
#cleaning
patients_df=patients.copy()
treatments_df=treatments.copy()
treatments_cut_df=treatments_cut.copy()
adverse_rxn_df=adverse_rxn.copy()


## 🧩 Handling Missing Values

Missing values are first represented explicitly as `No Data` for the patient table.

**Data Quality Dimension:** Completeness

**Issue Type:** Dirty data

The purpose is to make missing information visible and distinguish it from valid values. Later, specific fields are handled more appropriately where needed.


### Define
replace all missing values pf patients with no data

In [22]:
patients[patients_df['address'].isnull()]

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,contact,birthdate,weight,height,bmi
209,210,female,Lalita,Eldarkhanov,NaN,NaN,NaN,NaN,NaN,NaN,8/14/1950,143.4,62,26.2
219,220,male,Mỹ,Quynh,NaN,NaN,NaN,NaN,NaN,NaN,4/9/1978,237.8,69,35.1
230,231,female,Elisabeth,Knudsen,NaN,NaN,NaN,NaN,NaN,NaN,9/23/1976,165.9,63,29.4
234,235,female,Martina,Tománková,NaN,NaN,NaN,NaN,NaN,NaN,4/7/1936,199.5,65,33.2
242,243,male,John,O'Brian,NaN,NaN,NaN,NaN,NaN,NaN,2/25/1957,205.3,74,26.4
249,250,male,Benjamin,Mehler,NaN,NaN,NaN,NaN,NaN,NaN,10/30/1951,146.5,69,21.6
257,258,male,Jin,Kung,NaN,NaN,NaN,NaN,NaN,NaN,5/17/1995,231.7,69,34.2
264,265,female,Wafiyyah,Asfour,NaN,NaN,NaN,NaN,NaN,NaN,11/3/1989,158.6,63,28.1
269,270,female,Flavia,Fiorentino,NaN,NaN,NaN,NaN,NaN,NaN,10/9/1937,175.2,61,33.1
278,279,female,Generosa,Cabán,NaN,NaN,NaN,NaN,NaN,NaN,12/16/1962,124.3,69,18.4


In [23]:
patients_df.fillna('No Data',inplace=True)

/tmp/ipykernel_1908/757635791.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'No Data' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  patients_df.fillna('No Data',inplace=True)


## 🧮 Creating Derived Treatment Information

The treatment data contains starting and ending HbA1c values. The notebook calculates the change using:

`hba1c_change = hba1c_start - hba1c_end`

This creates a derived variable that can be used to understand the change between the two measurements.


In [24]:
treatments.head()

,given_name,surname,auralin,novodra,hba1c_start,hba1c_end,hba1c_change
0,veronika,jindrová,41u - 48u,-,7.63,7.20,NaN
1,elliot,richardson,-,40u - 45u,7.56,7.09,0.97
2,yukitaka,takenaka,-,39u - 36u,7.68,7.25,NaN
3,skye,gormanston,33u - 36u,-,7.97,7.62,0.35
4,alissa,montez,-,33u - 29u,7.78,7.46,0.32


In [25]:
treatments_df['hba1c_change']=treatments_df['hba1c_start']-treatments_df['hba1c_end']

In [26]:
treatments_df

,given_name,surname,auralin,novodra,hba1c_start,hba1c_end,hba1c_change
0,veronika,jindrová,41u - 48u,-,7.63,7.20,0.43
1,elliot,richardson,-,40u - 45u,7.56,7.09,0.47
2,yukitaka,takenaka,-,39u - 36u,7.68,7.25,0.43
3,skye,gormanston,33u - 36u,-,7.97,7.62,0.35
4,alissa,montez,-,33u - 29u,7.78,7.46,0.32
...,...,...,...,...,...,...,...
275,albina,zetticci,45u - 51u,-,7.93,7.73,0.20
276,john,teichelmann,-,49u - 49u,7.90,7.58,0.32
277,mathea,lillebø,23u - 36u,-,9.04,8.67,0.37
278,vallie,prince,31u - 38u,-,7.64,7.28,0.36


In [27]:
treatments_cut_df['hba1c_change']=treatments_cut_df['hba1c_start']-treatments_cut_df['hba1c_end']

In [28]:
treatments_cut_df

,given_name,surname,auralin,novodra,hba1c_start,hba1c_end,hba1c_change
0,jožka,resanovič,22u - 30u,-,7.56,7.22,0.34
1,inunnguaq,heilmann,57u - 67u,-,7.85,7.45,0.40
2,alwin,svensson,36u - 39u,-,7.78,7.34,0.44
3,thể,lương,-,61u - 64u,7.64,7.22,0.42
4,amanda,ribeiro,36u - 44u,-,7.85,7.47,0.38
...,...,...,...,...,...,...,...
65,rovzan,kishiev,32u - 37u,-,7.75,7.41,0.34
66,jakob,jakobsen,-,28u - 26u,7.96,7.51,0.45
67,bernd,schneider,48u - 56u,-,7.74,7.44,0.30
68,berta,napolitani,-,42u - 44u,7.68,7.21,0.47


In [29]:
treatments_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 280 entries, 0 to 279
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   given_name    280 non-null    object 
 1   surname       280 non-null    object 
 2   auralin       280 non-null    object 
 3   novodra       280 non-null    object 
 4   hba1c_start   280 non-null    float64
 5   hba1c_end     280 non-null    float64
 6   hba1c_change  280 non-null    float64
dtypes: float64(3), object(4)
memory usage: 15.4+ KB


## 🧹 Tidiness and Structural Cleaning

A **tidy dataset** generally has:

- each variable as a column
- each observation as a row
- each type of observational unit in an appropriate table

The patient `contact` column contains **two variables — phone number and email address — in a single column**.

**Data Quality Dimension:** Tidiness

**Issue Type:** Messy data

The solution is to extract the phone number and email into separate columns.


In [30]:
#tideness
#in patients table we will use regex to separate conatct number and email


### 📞 Separating Phone Number and Email

A regular expression is used to identify the phone number and email address inside the combined `contact` field.

This is a structural cleaning step because the original column stores multiple variables together.


In [31]:
import  re
def find_contact_details(text: str) -> tuple:
    # it the value is NaN, then return it
    if pd.isna(text):
        return np.nan

    # create the phone number pattern
    phone_number_pattern = re.compile(r"(\+[\d]{1,3}\s)?(\(?[\d]{3}\)?\s?-?[\d]{3}\s?-?[\d]{4})")
    # find the phone number from the value/text, as a result we will get a list
    phone_number  = re.findall(phone_number_pattern, text)

    # if length is 0, then the regex can't find any ph number, then define with NaN
    if len(phone_number) <= 0:
        phone_number = np.nan
    # if the country code is attached with the ph number, for that case, the first
    # element will be the country code and the 2nd element will be the actual ph
    # number. So, get that ph number
    elif len(phone_number) >= 2:
        phone_number = phone_number[1]
    # else, we will get the ph number. Grab it.
    else:
        phone_number = phone_number[0]

    # if we found the ph number (with/without country code), then remove that part from the actual value.
    # after removing the ph number, the remaining string might be the email address.
    possible_email_add = re.sub(phone_number_pattern, "", text).strip()

    # then return the ph number and the email address
    return phone_number, possible_email_add

In [32]:
patients_df.contact

,contact
0,951-719-9170ZoeWellish@superrito.com
1,PamelaSHill@cuvox.de+1 (217) 569-3204
2,402-363-6804JaeMDebord@gustr.com
3,PhanBaLiem@jourrapide.com+1 (732) 636-8246
4,334-515-7487TimNeudorf@cuvox.de
...,...
498,207-477-0579MustafaLindstrom@jourrapide.com
499,928-284-4492RumanBisliev@gustr.com
500,816-223-6007JinkedeKeizer@teleworm.us
501,ChidaluOnyekaozulu@jourrapide.com1 360 443 2060


In [33]:
find_contact_details('207-477-0579MustafaLindstrom@jourrapide.com')

(('', '207-477-0579'), 'MustafaLindstrom@jourrapide.com')

In [34]:
patients_df['phone']=patients_df['contact'].apply(lambda x:find_contact_details(x)).apply(lambda x:x[0])
patients_df['email']=patients_df['contact'].apply(lambda x:find_contact_details(x)).apply(lambda x:x[1])

In [35]:
patients_df.drop(columns='contact',inplace=True)

### 🔄 Combining Treatment Tables

The two treatment DataFrames are concatenated so that the treatment information can be processed together.

The next steps reshape the combined table into a more analysis-friendly structure.


In [36]:
treatments_df=pd.concat([treatments_df,treatments_cut_df])

In [37]:
treatments_df.shape

(350, 7)

### 🔄 Reshaping the Treatment Data

The treatment data is transformed from a wide structure into a longer structure using `melt()`.

**Data Quality Dimension:** Tidiness

**Issue Type:** Messy data

This makes treatment type a variable rather than spreading different treatment types across multiple columns.


In [38]:
treatments_df=treatments_df.melt(id_vars=['given_name','surname','hba1c_start','hba1c_end','hba1c_change'],var_name='type',value_name='dosage_range')

In [39]:
treatments_df=treatments_df[treatments_df.dosage_range!='-']

In [40]:
treatments_df['dosage_start']=treatments_df.dosage_range.str.split('-').str.get(0)

In [41]:
treatments_df['dosage_end']=treatments_df.dosage_range.str.split('-').str.get(1)

In [42]:
treatments_df.drop(columns='dosage_range',inplace=True)

In [43]:
treatments_df['dosage_start']=treatments_df['dosage_start'].str.replace('u','')

In [44]:
treatments_df['dosage_end']=treatments_df['dosage_end'].str.replace('u','')

In [45]:
treatments_df['dosage_start']=treatments_df['dosage_start'].astype('int')

In [46]:
treatments_df['dosage_end']=treatments_df['dosage_end'].astype('int')

### 🔗 Combining Treatment and Adverse-Reaction Information

The cleaned treatment data is merged with the adverse-reaction data using `given_name` and `surname`.

A left join is used so that treatment records are retained even when a matching adverse-reaction record is not available.


In [47]:
treatments_df=treatments_df.merge(adverse_rxn_df,how='left',on=['given_name','surname'])

## 🧹 Cleaning the Patient Data: Identified Issues

The following sections address specific problems found during assessment. For each issue, the notebook documents:

1. What was found
2. The data-quality dimension involved
3. Why it matters
4. How the value is cleaned

This turns the notebook from a sequence of code cells into a reproducible data-cleaning workflow.


In [48]:
treatments_df

,given_name,surname,hba1c_start,hba1c_end,hba1c_change,type,dosage_start,dosage_end,adverse_reaction
0,veronika,jindrová,7.63,7.20,0.43,auralin,41,48,NaN
1,skye,gormanston,7.97,7.62,0.35,auralin,33,36,NaN
2,sophia,haugen,7.65,7.27,0.38,auralin,37,42,NaN
3,eddie,archer,7.89,7.55,0.34,auralin,31,38,NaN
4,asia,woźniak,7.76,7.37,0.39,auralin,30,36,NaN
...,...,...,...,...,...,...,...,...,...
345,christopher,woodward,7.51,7.06,0.45,novodra,55,51,nausea
346,maret,sultygov,7.67,7.30,0.37,novodra,26,23,NaN
347,lixue,hsueh,9.21,8.80,0.41,novodra,22,23,injection site discomfort
348,jakob,jakobsen,7.96,7.51,0.45,novodra,28,26,hypoglycemia


### 🔎 Issue 1: Misspelled Patient Name

**Problem:** Patient ID 9 contains a misspelled given name (`dsvid`) instead of the expected name (`David`).

**Data Quality Dimension:** Accuracy

**Issue Type:** Dirty data

**Why it matters:** An incorrect name can cause problems when identifying or matching a patient record.

**Cleaning approach:** Use the patient's unique `patient_id` to target the affected record and correct the value.


In [49]:
#Issue 1: patient_id=9 has misssplled name 'dsvid' instead of david
patients_df.loc[patients_df.patient_id==9,'given_name']='David'

In [50]:
#correcting misspelled name
patients_df.loc[patients_df['patient_id'] == 9, 'given_name']

,given_name
8,David


### 🔎 Issue 2: Inconsistent State Representation

**Problem:** The `state` column contains both full state names and abbreviations, such as `California` and `CA`.

**Data Quality Dimension:** Consistency

**Issue Type:** Dirty data

**Why it matters:** The same category represented in different formats can lead to incorrect grouping, filtering, and counting.

**Cleaning approach:** Map full state names to their standard two-letter abbreviations.


In [51]:
#Issue 2 state column sometimes contains full name some times abbrivation
patients_df.state.value_counts()

,count
state,
California,36
TX,32
New York,25
CA,24
MA,22
NY,22
PA,18
GA,15
Illinois,14


In [52]:
state_abbreviations = {
    'California': 'CA',
    'Texas': 'TX',
    'New York': 'NY',
    'Massachusetts': 'MA',
    'Pennsylvania': 'PA',
    'Georgia': 'GA',
    'Illinois': 'IL',
    'Ohio': 'OH',
    'Florida': 'FL',
    'Michigan': 'MI',
    'Oklahoma': 'OK',
    'Louisiana': 'LA',
    'New Jersey': 'NJ',
    'Virginia': 'VA',
    'Mississippi': 'MS',
    'Wisconsin': 'WI',
    'Indiana': 'IN',
    'Minnesota': 'MN',
    'Tennessee': 'TN',
    'Alabama': 'AL',
    'North Carolina': 'NC',
    'Kentucky': 'KY',
    'Washington': 'WA',
    'Missouri': 'MO',
    'Idaho': 'ID',
    'Kansas': 'KS',
    'Nevada': 'NV',
    'South Carolina': 'SC',
    'Iowa': 'IA',
    'Connecticut': 'CT',
    'Maine': 'ME',
    'North Dakota': 'ND',
    'Nebraska': 'NE',
    'Rhode Island': 'RI',
    'Arkansas': 'AR',
    'Colorado': 'CO',
    'Arizona': 'AZ',
    'Maryland': 'MD',
    'Delaware': 'DE',
    'West Virginia': 'WV',
    'Oregon': 'OR',
    'South Dakota': 'SD',
    'Montana': 'MT',
    'Vermont': 'VT',
    'District of Columbia': 'DC',
    'Alaska': 'AK',
    'Wyoming': 'WY',
    'New Hampshire': 'NH',
    'New Mexico': 'NM',
    'No data': 'Unknown'
}
#apply changes to the df
patients_df['state']=patients_df['state'].apply(lambda x:state_abbreviations.get(x,x))
# Final check of the dataframe
print(patients_df['state'].value_counts())

state
CA         60
NY         47
TX         32
IL         24
MA         22
FL         22
PA         18
GA         15
OH         14
OK         13
MI         13
LA         13
No Data    12
NJ         12
VA         11
WI         10
MS         10
TN          9
IN          9
AL          9
MN          9
WA          8
KY          8
NC          8
MO          7
NV          6
KS          6
ID          6
NE          6
SC          5
CT          5
IA          5
AR          4
RI          4
ME          4
CO          4
AZ          4
ND          4
OR          3
MD          3
SD          3
DE          3
WV          3
MT          2
VT          2
DC          2
NM          1
WY          1
AK          1
NH          1
Name: count, dtype: int64


### 🔎 Issue 3: ZIP Code Format

**Problem:** ZIP codes contain values with different lengths and decimal representations. Some ZIP codes have four digits, even though a standard US ZIP code is represented using five digits.

**Data Quality Dimension:** Validity

**Issue Type:** Dirty data

**Why it matters:** ZIP codes should follow a consistent format. Leading zeros can also be lost when ZIP codes are stored as numbers.

**Cleaning approach:** Remove the unnecessary decimal representation and use zero-padding so valid ZIP codes have five characters.


In [53]:
#Issue 3:Zip code col has entries with 4 digit -validity
#validate and correct zip code format
patients_df['zip_code'].apply(str).str.len().value_counts()

,count
zip_code,
7,454
6,49


### 🛠️ Cleaning the ZIP Code

The cleaning function:

1. Handles the missing-value label `No Data`
2. Converts numeric-looking values to a number
3. Removes the decimal component
4. Converts the result back to a string
5. Uses `zfill(5)` to restore leading zeros where required

ZIP codes are ultimately better treated as **text**, because they are identifiers rather than quantities used for mathematical calculations.


In [54]:
# Data is in string but decimal format -  Need to remove decimal part and fill zeros upfrom to make it in 5 digit format
# Function to clean the zip code
def clean_zip_code(zip_code):
  if zip_code =='No Data':
    return 'NA'
  zip_code=int(float(zip_code))#remove decimal part
  return str(zip_code).zfill(5)

In [55]:
#Apply the function to the zip_code column
patients_df['zip_code'].apply(clean_zip_code).str.len().value_counts()

,count
zip_code,
5,491
2,12


In [56]:
# Apply the function to the zip_code column
patients_df['zip_code'] = patients_df['zip_code'].apply(clean_zip_code)

### 🔎 Issue 4: Missing Patient Information

**Problem:** 12 patients have missing values across address, city, state, ZIP code, country, and contact-related fields.

**Data Quality Dimension:** Completeness

**Issue Type:** Dirty data

**Why it matters:** Missing information reduces the completeness of patient records and may limit downstream analysis.

**Cleaning approach:** The notebook replaces the remaining missing location/contact fields with `Unknown` so the absence of information is represented explicitly.


In [57]:
# Issue 4: data missing for 12 patients in address,city, state,zip_code ,country, contact -completion
patients_df[patients_df.isnull().any(axis=1)]

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,birthdate,weight,height,bmi,phone,email
209,210,female,Lalita,Eldarkhanov,No Data,No Data,No Data,NA,No Data,8/14/1950,143.4,62,26.2,NaN,No Data
219,220,male,Mỹ,Quynh,No Data,No Data,No Data,NA,No Data,4/9/1978,237.8,69,35.1,NaN,No Data
230,231,female,Elisabeth,Knudsen,No Data,No Data,No Data,NA,No Data,9/23/1976,165.9,63,29.4,NaN,No Data
234,235,female,Martina,Tománková,No Data,No Data,No Data,NA,No Data,4/7/1936,199.5,65,33.2,NaN,No Data
242,243,male,John,O'Brian,No Data,No Data,No Data,NA,No Data,2/25/1957,205.3,74,26.4,NaN,No Data
249,250,male,Benjamin,Mehler,No Data,No Data,No Data,NA,No Data,10/30/1951,146.5,69,21.6,NaN,No Data
257,258,male,Jin,Kung,No Data,No Data,No Data,NA,No Data,5/17/1995,231.7,69,34.2,NaN,No Data
264,265,female,Wafiyyah,Asfour,No Data,No Data,No Data,NA,No Data,11/3/1989,158.6,63,28.1,NaN,No Data
269,270,female,Flavia,Fiorentino,No Data,No Data,No Data,NA,No Data,10/9/1937,175.2,61,33.1,NaN,No Data
278,279,female,Generosa,Cabán,No Data,No Data,No Data,NA,No Data,12/16/1962,124.3,69,18.4,NaN,No Data


In [58]:
patients_df[['address', 'city', 'state', 'zip_code', 'country', 'phone']] = patients_df[['address', 'city', 'state', 'zip_code', 'country', 'phone']].fillna('Unknown')

In [59]:
patients_df[patients_df.isnull().any(axis=1)]

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,birthdate,weight,height,bmi,phone,email


### 🔎 Issue 5: Incorrect Data Types

**Problem:** Some columns have data types that do not match their intended meaning.

Examples include:

- `assigned_sex` should behave as a categorical variable
- `zip_code` should be treated as text rather than a numeric measurement
- `birthdate` should be represented as a datetime value

**Data Quality Dimension:** Validity

**Issue Type:** Dirty data

Correct data types make filtering, grouping, sorting, and analysis more reliable.


In [60]:
# Issue 5: incorrect data type assigned to sex, zip code, birthdate -validity
patients_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   patient_id    503 non-null    int64  
 1   assigned_sex  503 non-null    object 
 2   given_name    503 non-null    object 
 3   surname       503 non-null    object 
 4   address       503 non-null    object 
 5   city          503 non-null    object 
 6   state         503 non-null    object 
 7   zip_code      503 non-null    object 
 8   country       503 non-null    object 
 9   birthdate     503 non-null    object 
 10  weight        503 non-null    float64
 11  height        503 non-null    int64  
 12  bmi           503 non-null    float64
 13  phone         503 non-null    object 
 14  email         503 non-null    object 
dtypes: float64(2), int64(2), object(11)
memory usage: 59.1+ KB


In [61]:
patients['assigned_sex']=patients_df['assigned_sex'].astype('category')
patients_df['zip_code']=patients_df['zip_code'].astype('str')
patients_df['birthdate'] = pd.to_datetime(patients_df['birthdate'], errors='coerce')
patients_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   patient_id    503 non-null    int64         
 1   assigned_sex  503 non-null    object        
 2   given_name    503 non-null    object        
 3   surname       503 non-null    object        
 4   address       503 non-null    object        
 5   city          503 non-null    object        
 6   state         503 non-null    object        
 7   zip_code      503 non-null    object        
 8   country       503 non-null    object        
 9   birthdate     503 non-null    datetime64[ns]
 10  weight        503 non-null    float64       
 11  height        503 non-null    int64         
 12  bmi           503 non-null    float64       
 13  phone         503 non-null    object        
 14  email         503 non-null    object        
dtypes: datetime64[ns](1), float64(2), int64(

### 🔎 Issue 6: Duplicate Patient Names

**Problem:** Multiple records contain the same `given_name` and `surname` combination.

**Data Quality Dimension:** Uniqueness / Accuracy

**Issue Type:** Potential duplicate data

The notebook investigates these repeated names and removes duplicate name combinations during cleaning.

**Important:** In real-world data, identical names alone are not sufficient proof that two people are the same. A unique identifier such as `patient_id` should normally be used to confirm duplicates.


In [62]:
# Issue 6: duplicate entries by the name of John Doe accuracy
patients_df[patients_df.duplicated(subset=['given_name', 'surname'])]

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,birthdate,weight,height,bmi,phone,email
229,230,male,John,Doe,123 Main Street,New York,NY,12345,United States,1975-01-01,180.0,72,24.4,"(, 1234567890)",johndoe@email.com
237,238,male,John,Doe,123 Main Street,New York,NY,12345,United States,1975-01-01,180.0,72,24.4,"(, 1234567890)",johndoe@email.com
244,245,male,John,Doe,123 Main Street,New York,NY,12345,United States,1975-01-01,180.0,72,24.4,"(, 1234567890)",johndoe@email.com
251,252,male,John,Doe,123 Main Street,New York,NY,12345,United States,1975-01-01,180.0,72,24.4,"(, 1234567890)",johndoe@email.com
277,278,male,John,Doe,123 Main Street,New York,NY,12345,United States,1975-01-01,180.0,72,24.4,"(, 1234567890)",johndoe@email.com


In [63]:
#remove duplicates
patients_df = patients_df.drop_duplicates(subset=['given_name', 'surname'])

### 🔎 Issue 7: Suspicious Weight Value

**Problem:** One patient has a recorded weight of approximately 48 pounds.

**Data Quality Dimension:** Accuracy

**Issue Type:** Dirty data

The value is investigated as a potential data-entry error because it is inconsistent with the surrounding patient measurements.

The next cells use the available weight and BMI information to estimate a more plausible height-related value.


In [64]:
# Issue 7: one patient has weight = 48 pounds accuracy
patients_df.loc[(patients_df['weight'] == 48)]

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,birthdate,weight,height,bmi,phone,email


### 🔎 Issue 8: Suspicious Height Value

**Problem:** One patient has a recorded height of 27 inches.

**Data Quality Dimension:** Accuracy

**Issue Type:** Dirty data

The value is investigated because it appears inconsistent with the patient's other measurements. The notebook uses the recorded weight and BMI to estimate the corresponding height.


In [65]:
# Issue 8: one patient has height = 27 inches accuracy

patients_df.loc[(patients_df['height'] == 27)]

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,birthdate,weight,height,bmi,phone,email
4,5,male,Tim,Neudorf,1428 Turkey Pen Lane,Dothan,AL,36303,United States,1928-02-18,192.3,27,26.1,"(, 334-515-7487)",TimNeudorf@cuvox.de


In [66]:
# Given values
weight_lb = 192.3
bmi = 26.1

# Convert weight from pounds to kilograms
weight_kg = weight_lb * 0.453592

# Calculate height in meters
height_m = (weight_kg / bmi)**0.5

# Convert height from meters to inches
height_inches = height_m * 39.3701

print(f"Corrected height in inches: {height_inches:.2f}")

Corrected height in inches: 71.97


In [67]:
# Update the height for the rows where height is 27 inches
patients_df.loc[patients_df['height'] == 27, 'height'] = 71.97
patients_df.loc[(patients_df['height'] == 27)]

/tmp/ipykernel_1908/2540299318.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '71.97' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  patients_df.loc[patients_df['height'] == 27, 'height'] = 71.97


,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,birthdate,weight,height,bmi,phone,email


## 💊 Treatment Data Cleaning

The treatment dataset is assessed separately after the patient data has been cleaned.

The main issues addressed here are **consistency** and **duplicate records**.


## Treatment Data


In [68]:
treatments_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   given_name        350 non-null    object 
 1   surname           350 non-null    object 
 2   hba1c_start       350 non-null    float64
 3   hba1c_end         350 non-null    float64
 4   hba1c_change      350 non-null    float64
 5   type              350 non-null    object 
 6   dosage_start      350 non-null    int64  
 7   dosage_end        350 non-null    int64  
 8   adverse_reaction  35 non-null     object 
dtypes: float64(3), int64(2), object(4)
memory usage: 24.7+ KB


### 🔎 Treatment Issue 1: Inconsistent Name Formatting

**Problem:** `given_name` and `surname` values are not consistently capitalized.

**Data Quality Dimension:** Consistency

**Issue Type:** Dirty data

**Why it matters:** Inconsistent capitalization can cause matching and grouping problems, especially when names are used to merge or compare datasets.

**Cleaning approach:** Standardize names using title case.


In [69]:
# Issue 1: given_name and surname col is is all lower case `consistency`
treatments_df['given_name']=treatments_df['given_name'].str.title()
treatments_df['surname']=treatments_df['surname'].str.title()

In [70]:
treatments_df.head()

,given_name,surname,hba1c_start,hba1c_end,hba1c_change,type,dosage_start,dosage_end,adverse_reaction
0,Veronika,Jindrová,7.63,7.20,0.43,auralin,41,48,NaN
1,Skye,Gormanston,7.97,7.62,0.35,auralin,33,36,NaN
2,Sophia,Haugen,7.65,7.27,0.38,auralin,37,42,NaN
3,Eddie,Archer,7.89,7.55,0.34,auralin,31,38,NaN
4,Asia,Woźniak,7.76,7.37,0.39,auralin,30,36,NaN


### 🔎 Treatment Issue 2: Duplicate Treatment Record

**Problem:** A duplicate treatment entry is identified for Joseph Day.

**Data Quality Dimension:** Uniqueness / Accuracy

**Issue Type:** Dirty data

Duplicate treatment records can lead to double-counting and incorrect analysis, so the repeated record is investigated before removal.


In [71]:
# Issue 2:  1 duplicate entry by the name Joseph day `accuracy`
treatments_df[treatments_df.duplicated(subset=['given_name', 'surname'], )]

,given_name,surname,hba1c_start,hba1c_end,hba1c_change,type,dosage_start,dosage_end,adverse_reaction
62,Joseph,Day,7.7,7.19,0.51,auralin,29,36,hypoglycemia


In [72]:
treatments_df[treatments_df['given_name']=='Joseph']

,given_name,surname,hba1c_start,hba1c_end,hba1c_change,type,dosage_start,dosage_end,adverse_reaction
5,Joseph,Day,7.70,7.19,0.51,auralin,29,36,hypoglycemia
62,Joseph,Day,7.70,7.19,0.51,auralin,29,36,hypoglycemia
167,Joseph,Tucker,7.67,7.30,0.37,auralin,48,56,NaN


### ✅ Removing the Duplicate Treatment Record

The duplicate Joseph Day record is removed while keeping the first occurrence.

After this step, the treatment table contains a cleaner set of records for further analysis.


In [73]:
# Remove duplicate entry by the name Joseph Day
treatments_df = treatments_df.drop_duplicates(subset=['given_name', 'surname'], keep='first')
treatments_df[treatments_df['given_name']=='Joseph']

,given_name,surname,hba1c_start,hba1c_end,hba1c_change,type,dosage_start,dosage_end,adverse_reaction
5,Joseph,Day,7.70,7.19,0.51,auralin,29,36,hypoglycemia
167,Joseph,Tucker,7.67,7.30,0.37,auralin,48,56,NaN


# 📌 Summary

This notebook demonstrates a complete data-cleaning workflow:

### 🔎 Assessment
- Inspected sample records
- Checked data types and missing values
- Investigated duplicates
- Examined numerical distributions

### 🧹 Cleaning
- Handled missing values
- Corrected inaccurate values
- Standardized state and name formats
- Cleaned ZIP codes
- Corrected data types
- Removed duplicate records

### 🧩 Tidiness
- Split combined contact information into separate variables
- Reshaped treatment data using `melt()`
- Combined related datasets using concatenation and merging

### 📚 Data Quality Dimensions

The notebook demonstrates issues related to:

**Completeness · Validity · Accuracy · Consistency · Uniqueness · Tidiness**

The key principle is:

> **Assess the data first, identify the quality problem, decide on an appropriate cleaning strategy, and then apply the transformation.**
